## Ollama + Tool calling + Validation

In [1]:
from dataclasses import dataclass, field
import logging

import ollama


logging.basicConfig(level=logging.INFO)

@dataclass
class ToolCall:
    name: str
    arguments: dict


@dataclass
class LLMResponse:
    content: str
    total_time_s: float
    load_time_s: float
    generated_tokens: int
    tokens_per_second: float
    tool_calls: list[ToolCall] = field(default_factory=list)


class LLMBackend:
    def generate(self, messages):
        raise NotImplementedError


def add(a: float, b: float) -> float:
    """
    Add two numbers.
    Use this tool only when addition is needed.

    Args:
        a: First number.
        b: Second number.

    Returns:
        The sum of a and b.
    """
    logging.info("Adding %f and %f inside the add tool: %f", a, b, a + b)
    return a + b



def div(a: float, b: float) -> float:
    """
    Divide two numbers.
    Use this tool only when division is needed.

    Args:
        a: First number.
        b: Second number.

    Returns:
        The result of dividing a by b.
    """
    if b == 0:
        raise ValueError("Division by zero is not allowed.")
    
    logging.info("Dividing %f by %f inside the div tool: %f", a, b, a / b)
    return a / b


def save_to_file(filename: str, content: str) -> str:
    """
    Save content to a file.
    Use this tool only when saving content to a file is needed.

    Args:
        filename: The name of the file to save the content to.
        content: The content to be saved.  

    Returns:
        A message indicating the result of the save operation.
    """
    with open(filename, "w") as f:
        f.write(content)
    logging.info("Content saved to file: %s", filename)
    return f"Content saved to {filename}"


functions = {
    "add": add,
    "div": div,
    "save_to_file": save_to_file,
}

tools = list(functions.values())
    

class OllamaBackend(LLMBackend):
    def __init__(self, model, tools: list=None):
        self.model = model

    def generate(self, messages: list[dict], tools: list=None) -> LLMResponse:
        output = ollama.chat(
            model=self.model,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "Use a tool only when its documented purpose matches a "
                        "necessary step in answering the user's request. "
                        "Tools are optional; their availability does not mean "
                        "you must use them. "
                        "If no available tool is appropriate, answer directly "
                        "when you can. "
                        "Check that tool results are relevant before using them "
                        "in your answer. "
                        "Availability of a tool does not mean it is always the best choice. "
                    ),
                },
                *messages,
            ],
            tools=tools,
            think=False,
            stream=False,
            options={"num_ctx": 8192, "num_predict": 800}
        )
        
        content = output['message']['content']
        total_time_s = output['total_duration'] / 1e9
        load_time_s = output['load_duration'] / 1e9
        generated_tokens = output.eval_count
        generation_time = output.eval_duration / 1e9
        tool_calls = [
            ToolCall(name=call.function['name'], arguments=call.function['arguments'])
            for call in (output.message.tool_calls or [])
        ]

        tokens_per_second = (
            generated_tokens / generation_time
            if generation_time > 0 else 0
        )

        if not content and not tool_calls:
            raise ValueError("No content returned from the model.")
        return LLMResponse(
            content=content,
            total_time_s=total_time_s,
            load_time_s=load_time_s,
            generated_tokens=generated_tokens,
            tokens_per_second=tokens_per_second,
            tool_calls=tool_calls
        )



In [ ]:
from pydantic import BaseModel, Field, field_validator, ValidationError


# We want to validate the inputs to the tools
class AddInput(BaseModel):
    a: float
    b: float

class DivInput(BaseModel):
    a: float
    b: float

    @field_validator('b')
    def b_must_not_be_zero(cls, v):
        if v == 0:
            raise ValueError("Second number cannot be zero.")
        return v

class SaveToFileInput(BaseModel):
    filename: str
    content: str

# Dictionary mapping tool names to their corresponding input models 
# (used for validating tool call arguments)
input_models = {
    "add": AddInput,
    "div": DivInput,
    "save_to_file": SaveToFileInput,
}


MODEL_NAME = "gemma4:e2b"  # "qwen3.5:2b"
llm = OllamaBackend(model=MODEL_NAME)
messages = []
MAX_HISTORY_MESSAGES = 8
MAX_TOOL_ROUNDS = 5
tool_round = 0


while True:
    user_input = input("You: ").strip()
    if user_input.lower() in {"exit", "quit"}:
        break

    tool_round = 0
    messages.append({"role": "user", "content": user_input})

    remaining_messages = messages[-MAX_HISTORY_MESSAGES:]
    output = llm.generate(messages=remaining_messages, tools=tools)

    # Multi-step tool-using agent loop:
    # Handle tool calls in a loop until there are no more tool calls or the maximum number of rounds is reached
    while output.tool_calls:
        tool_round += 1

        if tool_round > MAX_TOOL_ROUNDS:
            raise RuntimeError("Maximum tool rounds reached. Stopping tool calls.")

        # Append the model's response to the messages, 
        # including any tool name and arguments for the tool calls
        messages.append({
            "role": "assistant",
            "content": output.content,
            "tool_calls": [
                {
                    "type": "function",
                    "function": {
                        "name": call.name,
                        "arguments": call.arguments,
                    },
                }
                for call in output.tool_calls
            ],
        })

        # Iterate through each tool call and execute the corresponding function,
        # And append the tool output to the messages
        for tool_call in output.tool_calls:
            # 
            tool = functions.get(tool_call.name, None)
            if tool is None:
                raise ValueError(f"Tool '{tool_call.name}' not found.")

            # Select the appropriate input model for the tool call
            input_model = input_models[tool_call.name]

            try:
                # Validate the tool call arguments using the input model
                validated = input_model.model_validate(tool_call.arguments)
            except ValidationError as exc:
                # If validation fails, set the tool output to an error message
                tool_output = f"Invalid tool arguments: {exc}"
            else:
                # Execute the tool with the validated arguments
                tool_output = tool(**validated.model_dump())

            # Append the tool output to the messages
            messages.append({
                "role": "tool",
                "tool_name": tool_call.name,
                "content": str(tool_output),
            })
        # Generate a new response from the model after executing the tools, 
        # And including the tool outputs in the messages for context
        output = llm.generate(messages=messages, tools=tools)

    messages.append({"role": "assistant", "content": output.content})
    if output.content:
        print(f"Assistant: {output.content}")
    print(f"Total time (s): {output.total_time_s}")
    print(f"Load time (s): {output.load_time_s}")
    print(f"Generated tokens: {output.generated_tokens}")
    print(f"Tokens per second: {output.tokens_per_second}")
    

INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:root:Adding 4.000000 and 6.000000 inside the add tool: 10.000000
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Assistant: 10
Total time (s): 0.137836459
Load time (s): 0.00152825
Generated tokens: 3
Tokens per second: 87.02715247157113


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:root:Dividing 8.000000 by 2.000000 inside the div tool: 4.000000
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Assistant: 4
Total time (s): 0.110962166
Load time (s): 0.000775791
Generated tokens: 2
Tokens per second: 118.63803535413453


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:root:Dividing 10.000000 by 3.000000 inside the div tool: 3.333333
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:root:Adding 3.333333 and 2.000000 inside the add tool: 5.333333
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


...

Assistant: The capital of Iran is Tehran.
Total time (s): 0.483437084
Load time (s): 0.001510917
Generated tokens: 8
Tokens per second: 70.36059806508355


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Assistant: Please provide the content you would like me to save to `answer.log`.
Total time (s): 0.624658333
Load time (s): 0.006252375
Generated tokens: 17
Tokens per second: 66.64236180530237


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:root:Content saved to file: answer.log
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


ValueError: No content returned from the model.